# Hệ thống Gợi ý Việc làm sử dụng ChromaDB (Vector Database)

Notebook này là phiên bản nâng cấp của `job_post_recommendation_ml_v2.ipynb`.
Thay vì tính toán ma trận tương đồng (Matrix Factorization) trên RAM, chúng ta sử dụng **ChromaDB** để lưu trữ và truy xuất vector.

**Ưu điểm:**
- Tốc độ truy vấn nhanh hơn.
- Tiết kiệm RAM.
- Dễ dàng tích hợp vào Flask Server.
- Persistency (Dữ liệu không bị mất khi tắt server).

## 1. Setup & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize
import re
import torch
import os
import shutil
import warnings

warnings.filterwarnings('ignore')

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 2. Load Data & Preprocessing (Tái sử dụng từ V2)
Chúng ta giữ nguyên logic xử lý dữ liệu và NLP từ phiên bản trước.

In [ ]:
# 2.1 Load Data
candidates = pd.read_csv("../data/csv_data/candidates.csv")
users = pd.read_csv("../data/csv_data/users.csv")
candidates_skills = pd.read_csv("../data/csv_data/candidates_skills.csv")
skills = pd.read_csv("../data/csv_data/skills.csv")

job_posts = pd.read_csv("../data/csv_data/job_posts.csv")
job_posts_skills = pd.read_csv("../data/csv_data/job_posts_skills.csv")
industries = pd.read_csv("../data/csv_data/industries.csv")
levels = pd.read_csv("../data/csv_data/levels.csv")
job_types = pd.read_csv("../data/csv_data/job_types.csv")

# Merge Data (Candidates)
candidates_full = candidates.merge(users[['id', 'full_name', 'summary']], on='id', how='left')
candidate_skills_grouped = candidates_skills.merge(skills[['id', 'name']], left_on='skills_id', right_on='id', how='left')
candidate_skills_agg = candidate_skills_grouped.groupby('candidate_id')['name'].apply(lambda x: ', '.join(x.dropna())).reset_index()
candidate_skills_agg.columns = ['candidate_id', 'skills_list']
df_candidates = candidates_full.merge(candidate_skills_agg, left_on='id', right_on='candidate_id', how='left')
df_candidates = df_candidates[['id', 'summary', 'education', 'expect_salary', 'skills_list']].copy()
df_candidates.columns = ['candidate_id', 'summary', 'education', 'expect_salary', 'skills']

# Merge Data (Jobs)
job_posts_full = job_posts.merge(industries[['id', 'name']], left_on='industry_id', right_on='id', how='left').rename(columns={'name': 'industry'})
job_posts_full = job_posts_full.merge(levels[['id', 'name']], left_on='level_id', right_on='id', how='left').rename(columns={'name': 'level'})
job_posts_full = job_posts_full.merge(job_types[['id', 'name']], left_on='job_type_id', right_on='id', how='left').rename(columns={'name': 'job_type'})
job_skills_grouped = job_posts_skills.merge(skills[['id', 'name']], left_on='skills_id', right_on='id', how='left')
job_skills_agg = job_skills_grouped.groupby('job_post_id')['name'].apply(lambda x: ', '.join(x.dropna())).reset_index()
job_skills_agg.columns = ['job_post_id', 'skills_list']
df_jobs = job_posts_full.merge(job_skills_agg, left_on='id', right_on='job_post_id', how='left')
df_jobs = df_jobs[['id', 'title', 'description', 'salary', 'industry', 'level', 'job_type', 'skills_list']].copy()
df_jobs.columns = ['job_id', 'title', 'description', 'salary', 'industry', 'level', 'job_type', 'skills']

# Fillna
df_candidates.fillna('', inplace=True)
df_candidates['expect_salary'] = pd.to_numeric(df_candidates['expect_salary'], errors='coerce').fillna(0)
df_jobs.fillna('', inplace=True)
df_jobs['salary'] = pd.to_numeric(df_jobs['salary'], errors='coerce').fillna(0)

print(f"Candidates: {len(df_candidates)}")
print(f"Jobs: {len(df_jobs)}")

In [ ]:
# 2.2 NLP Preprocessing Functions (Reuse)
STOPWORDS_PATH = "../data/nlp/vietnamese-stopwords.txt"
def load_vietnamese_stopwords(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return set(line.strip() for line in f if line.strip())
    except:
        return set()

vietnamese_stopwords = load_vietnamese_stopwords(STOPWORDS_PATH)

def preprocess_vietnamese_text(text):
    if not text or pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'[^a-zàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ0-9\s]', ' ', text)
    tokens = word_tokenize(text, format="text").split()
    clean_tokens = [t for t in tokens if t not in vietnamese_stopwords and not t.isdigit() and len(t) > 1]
    return " ".join(clean_tokens)

# Apply Preprocessing
print("Preprocessing Candidates...")
df_candidates['semantic_text'] = (df_candidates['summary'] + " " + df_candidates['education']).apply(preprocess_vietnamese_text)

print("Preprocessing Jobs...")
df_jobs['semantic_text'] = (df_jobs['title'] + " " + df_jobs['description']).apply(preprocess_vietnamese_text)

## 3. Vectorization & ChromaDB Setup (MỚI)
Thay vì tính toán ma trận, chúng ta sẽ:
1. Load model SentenceTransformer.
2. Khởi tạo ChromaDB.
3. Index dữ liệu Job Posts vào ChromaDB.

In [ ]:
# 3.1 Load Embedding Model
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
embedding_model = SentenceTransformer(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model.to(device)
print(f"Model loaded: {model_name} on {device}")

In [ ]:
# 3.2 Setup ChromaDB
DB_PATH = "../chroma_db_data"

# [QUAN TRỌNG] Nếu bạn đổi model embedding, kích thước vector sẽ thay đổi (ví dụ từ 768 -> 384)
# ChromaDB không cho phép add vector khác kích thước vào collection cũ.
# Đoạn code này sẽ tự động xóa DB cũ nếu cần thiết (Cẩn thận khi dùng trong production!)
RESET_DB = True 

if RESET_DB and os.path.exists(DB_PATH):
    print(f"Deleting old DB at {DB_PATH} to avoid dimension mismatch...")
    try:
        shutil.rmtree(DB_PATH)
    except Exception as e:
        print(f"Could not delete old DB: {e}. Please delete manually if you encounter errors.")

# Khởi tạo Client
chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Tạo collection (nếu chưa có) hoặc lấy collection cũ
# Dùng cosine distance (mặc định của Chroma là L2, nhưng cosine tốt hơn cho text similarity)
collection = chroma_client.get_or_create_collection(
    name="job_posts",
    metadata={"hnsw:space": "cosine"} 
)

print(f"Collection 'job_posts' ready. Current count: {collection.count()}")

In [ ]:
# 3.3 Indexing Data (Chỉ chạy 1 lần hoặc khi có dữ liệu mới)
# Nếu collection đã có dữ liệu, bạn có thể muốn xóa đi làm lại: collection.delete(ids=...)

if collection.count() == 0: # Chỉ index nếu DB rỗng
    print("Start Indexing Job Posts...")
    
    # Tạo embeddings cho toàn bộ jobs
    # Lưu ý: Nếu dữ liệu lớn (>10k), nên chia batch để encode
    job_embeddings = embedding_model.encode(df_jobs['semantic_text'].tolist(), show_progress_bar=True)
    
    ids = df_jobs['job_id'].astype(str).tolist()
    documents = df_jobs['title'].tolist()
    
    # Metadata: Lưu các trường cần thiết cho Re-ranking
    metadatas = []
    for _, row in df_jobs.iterrows():
        metadatas.append({
            "salary": float(row['salary']) if row['salary'] else 0.0,
            "skills": str(row['skills']),
            "industry": str(row['industry']),
            "level": str(row['level'])
        })
        
    # Insert vào ChromaDB (Batch size mặc định của Chroma xử lý tốt)
    collection.add(
        ids=ids,
        embeddings=job_embeddings.tolist(),
        metadatas=metadatas,
        documents=documents
    )
    print(f"Indexed {len(ids)} jobs successfully!")
else:
    print("Data already indexed. Skipping...")

## 4. Recommendation Logic (Retrieve & Re-rank)

Quy trình:
1. **Retrieve**: Dùng vector của Candidate tìm Top 50 Jobs tương đồng nhất về ngữ nghĩa (Semantic).
2. **Re-rank**: Tính điểm chi tiết (Salary, Skill Match) cho 50 Jobs này và sắp xếp lại.

In [ ]:
def calculate_final_score(semantic_score, job_meta, candidate_profile):
    """
    Tính điểm tổng hợp dựa trên trọng số (Alpha, Beta, Gamma)
    """
    # 1. Salary Score
    job_salary = job_meta['salary']
    cand_salary = candidate_profile.get('expect_salary', 0)
    salary_score = 0.0
    
    if cand_salary > 0 and job_salary > 0:
        # Nếu lương job >= lương mong đợi -> 1 điểm
        if job_salary >= cand_salary:
            salary_score = 1.0
        else:
            # Nếu thấp hơn, trừ điểm theo tỷ lệ
            diff_percent = (cand_salary - job_salary) / cand_salary
            salary_score = max(0, 1.0 - diff_percent)
    else:
        salary_score = 0.5 # Neutral nếu thiếu thông tin lương

    # 2. Skill Score (Set Intersection cơ bản)
    # Để tối ưu, có thể dùng embedding cho skill, nhưng ở đây dùng string matching cho nhanh
    job_skills = set([s.strip().lower() for s in job_meta['skills'].split(',') if s.strip()])
    cand_skills = set([s.strip().lower() for s in candidate_profile.get('skills', '').split(',') if s.strip()])
    
    skill_score = 0.0
    if len(job_skills) > 0 and len(cand_skills) > 0:
        match_count = len(job_skills.intersection(cand_skills))
        skill_score = match_count / len(job_skills) # Tỷ lệ đáp ứng yêu cầu của Job
    
    # 3. Combine Score
    ALPHA = 0.45 # Skill
    BETA = 0.40  # Semantic
    GAMMA = 0.15 # Salary
    
    final_score = (ALPHA * skill_score) + (BETA * semantic_score) + (GAMMA * salary_score)
    
    return final_score, skill_score, salary_score

def recommend_jobs(candidate_id, top_k=10):
    # 1. Get Candidate Info
    cand_row = df_candidates[df_candidates['candidate_id'] == candidate_id]
    if len(cand_row) == 0:
        return None
    
    cand_data = cand_row.iloc[0]
    
    # 2. Create Query Vector
    query_text = cand_data['semantic_text']
    query_vector = embedding_model.encode(query_text).tolist()
    
    # 3. Retrieve from ChromaDB (Lấy Top 50)
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=50,
        include=["metadatas", "documents", "distances"]
    )
    
    # 4. Re-rank
    ranked_jobs = []
    
    ids = results['ids'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]
    titles = results['documents'][0]
    
    for i in range(len(ids)):
        # Convert Cosine Distance to Similarity (1 - distance)
        # Lưu ý: Chroma trả về distance, càng nhỏ càng tốt. 
        semantic_score = 1 - distances[i]
        
        final_score, skill_score, salary_score = calculate_final_score(
            semantic_score, 
            metadatas[i], 
            cand_data
        )
        
        ranked_jobs.append({
            "job_id": ids[i],
            "title": titles[i],
            "final_score": final_score,
            "semantic_score": semantic_score,
            "skill_score": skill_score,
            "salary_score": salary_score,
            "salary": metadatas[i]['salary'],
            "industry": metadatas[i]['industry']
        })
        
    # Sort by Final Score
    ranked_jobs.sort(key=lambda x: x['final_score'], reverse=True)
    
    return pd.DataFrame(ranked_jobs[:top_k])

## 5. Demo & Testing

In [ ]:
# Chọn một candidate để test
sample_candidate_id = df_candidates.iloc[18]['candidate_id']
print(f"Testing recommendation for Candidate ID: {sample_candidate_id}")
print(f"Skills: {df_candidates.iloc[18]['skills']}")
print(f"Expected Salary: {df_candidates.iloc[18]['expect_salary']}")

recommendations = recommend_jobs(sample_candidate_id, top_k=10)

if recommendations is not None:
    print("\nTop 10 Recommendations:")
    display(recommendations)